# Rendu du décor « The Chase » sur Google Colab (repli)

Runtime : *Exécution → Modifier le type d'exécution → GPU T4*. Un compte Google suffit. Lancer *Exécution → Tout exécuter*. Les lignes de résultats s'affichent : réglage ; mode ; temps par image ; mémoire GPU.

# Rendu du décor « The Chase » sur GPU Kaggle

Activer Internet et GPU dans les paramètres du notebook. Aucune clé ni jeton n'est nécessaire. Les résultats s'affichent une ligne par réglage : réglage ; temps par image ; mémoire GPU.

In [1]:
import shutil, subprocess
if shutil.which("nvidia-smi"):
    print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"], capture_output=True, text=True).stdout)
    GPU_OK = True
else:
    print("Aucun GPU dans cette session : le rendu se fera sur CPU.")
    GPU_OK = False
print("GPU_OK", GPU_OK)

Tesla T4, 15360 MiB

GPU_OK True


In [2]:
import os, subprocess, tarfile, urllib.request
B='/content/blender'
if not os.path.exists(B):
    url='https://download.blender.org/release/Blender4.5/blender-4.5.15-linux-x64.tar.xz'
    urllib.request.urlretrieve(url,'/content/blender.tar.xz')
    with tarfile.open('/content/blender.tar.xz') as t: t.extractall('/content/')
    os.rename('/content/blender-4.5.15-linux-x64', B)
print('blender ok')

/tmp/ipykernel_746/3724370875.py:6: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  with tarfile.open('/content/blender.tar.xz') as t: t.extractall('/content/')


blender ok


In [3]:
import os, subprocess
if not os.path.exists('/content/depot'):
    subprocess.run(['git','clone','--depth','1','https://github.com/Dony1769/moteur-animation-eclipse','/content/depot'],check=True)
else:
    print("Le dépôt existe déjà dans /content/depot, étape de clonage ignorée.")

In [4]:
import json, os, urllib.request
RACINE = "/content/depot/assets/ph"
os.makedirs(f"{RACINE}/textures", exist_ok=True)
MODELES = {"namaqualand_cliff_01": "namaqualand_cliff_01", "rock_face_01": "rock_face_01",
           "rock_face_02": "rock_face_02", "sand_rocks_small_01": "sand_rocks_small_01"}

def telecharger(url, dst):
    if not os.path.exists(dst):
        req = urllib.request.Request(url, headers={'User-Agent': 'Mozilla/5.0'})
        with urllib.request.urlopen(req) as response, open(dst, 'wb') as out_file:
            out_file.write(response.read())

for nom in MODELES:
    req = urllib.request.Request(
        f"https://api.polyhaven.com/files/{nom}",
        headers={'User-Agent': 'Mozilla/5.0'}
    )
    f = json.loads(urllib.request.urlopen(req).read().decode('utf-8'))
    if "1k" in f.get("blend", {}):
        telecharger(f["blend"]["1k"]["blend"]["url"], f"{RACINE}/{nom}.blend")
    for court, cle, ext in (("diff", "Diffuse", "jpg"), ("nor_gl", "nor_gl", "exr"), ("rough", "Rough", "exr")):
        if cle in f and "1k" in f[cle] and ext in f[cle]["1k"]:
            telecharger(f[cle]["1k"][ext]["url"], f"{RACINE}/textures/{nom}_{court}_1k.{ext}")

telecharger("https://dl.polyhaven.org/file/ph-assets/HDRIs/exr/1k/belfast_sunset_puresky_1k.exr", f"{RACINE}/belfast_sunset_puresky_1k.exr")
print("assets ok")

assets ok


In [5]:
import os, subprocess, time, shutil
os.chdir("/content/depot")

# Correction du chemin vers l'exécutable réel et attribution des droits d'exécution
blender_bin = "/content/blender/blender"
if os.path.exists(blender_bin):
    os.chmod(blender_bin, 0o755)

GPU_OK = shutil.which("nvidia-smi") is not None
reglages = [(1280, 540, 16), (960, 405, 8), (640, 270, 8), (640, 270, 4)]
for rx, ry, spp in reglages:
    env = dict(os.environ, RX=str(rx), RY=str(ry), SPP=str(spp))
    env["DEVICE"] = "GPU" if GPU_OK else "CPU"
    env["BACKEND"] = "OPTIX"
    t = time.time()
    t_rendu = time.time()
    r = subprocess.run([blender_bin, "-b", "--factory-startup", "--python", "/content/depot/scenes/chase_a.py", "--", "/content/out"], env=env, capture_output=True, text=True)
    png = "/content/out/chase_a_decor.png"
    if not os.path.exists(png) or os.path.getmtime(png) < t_rendu:
        print("ECHEC", r.stderr[-400:])
    print([l for l in r.stdout.splitlines() if l.startswith("RENDU")][:1])
    mem = subprocess.run(["nvidia-smi", "--query-gpu=memory.used", "--format=csv,noheader"], capture_output=True, text=True).stdout.strip() if GPU_OK else "sans GPU"
    print(f"{rx}x{ry} {spp} ech. ; {env['DEVICE']} ; {time.time()-t:.1f} s ; {mem}")

["RENDU GPU OPTIX ['Tesla T4', 'Tesla T4']"]
1280x540 16 ech. ; GPU ; 19.6 s ; 0 MiB
["RENDU GPU OPTIX ['Tesla T4', 'Tesla T4']"]
960x405 8 ech. ; GPU ; 11.5 s ; 0 MiB
["RENDU GPU OPTIX ['Tesla T4', 'Tesla T4']"]
640x270 8 ech. ; GPU ; 5.3 s ; 0 MiB
["RENDU GPU OPTIX ['Tesla T4', 'Tesla T4']"]
640x270 4 ech. ; GPU ; 5.3 s ; 0 MiB


# Étape 5 : clip Mixamo (course sur place, 4 s)

Facultatif. Il faut les deux fichiers Mixamo (personnage avec peau et animation *Running* sans peau, format FBX binaire). Ils ne sont pas dans le dépôt : la cellule suivante demande de les déposer.
Sans eux, la cellule ne fait rien et le notebook se termine normalement.
Résultat : `/content/out_mixamo/course_sur_place_4s.mp4` (96 images, 640×270, 24 images/s).

In [9]:
import os
DOSSIER_MIXAMO = "/content/depot/assets/mixamo"
REQUIS = ["Personnage_Maria.fbx", "Running.fbx"]
os.makedirs(DOSSIER_MIXAMO, exist_ok=True)
manquants = [f for f in REQUIS if not os.path.exists(os.path.join(DOSSIER_MIXAMO, f))]
if manquants:
    try:
        from google.colab import files
        print("À déposer dans la fenêtre :", manquants)
        envoyes = files.upload()
        for nom, contenu in envoyes.items():
            nom_destination = nom
            nom_lower = nom.lower()

            # Gestion du personnage Maria
            if "maria" in nom_lower and "Personnage_Maria.fbx" in manquants:
                nom_destination = "Personnage_Maria.fbx"
                print(f"Association : '{nom}' renommé en 'Personnage_Maria.fbx'")

            # Gestion du fichier d'animation (Running / running)
            elif "running" in nom_lower and "Running.fbx" in manquants:
                nom_destination = "Running.fbx"
                print(f"Association : '{nom}' renommé en 'Running.fbx'")

            if nom_destination in REQUIS:
                with open(os.path.join(DOSSIER_MIXAMO, nom_destination), "wb") as fichier:
                    fichier.write(contenu)
    except ImportError:
        print("google.colab absent : hors Colab, dépôt impossible ici.")
    except Exception as e:
        print("Impossible d'ouvrir la fenêtre de dépôt :", e)
manquants = [f for f in REQUIS if not os.path.exists(os.path.join(DOSSIER_MIXAMO, f))]
MIXAMO_OK = not manquants
print("MIXAMO_OK", MIXAMO_OK, "manquants", manquants)

À déposer dans la fenêtre : ['Personnage_Maria.fbx']


Saving Personnage_Maria.fbx to Personnage_Maria.fbx
Saving Running.fbx to Running (1).fbx
Association : 'Personnage_Maria.fbx' renommé en 'Personnage_Maria.fbx'
MIXAMO_OK True manquants []


In [10]:
import os, subprocess, time, shutil
SORTIE = "/content/out_mixamo"
os.makedirs(SORTIE, exist_ok=True)
if not MIXAMO_OK:
    print("Fichiers Mixamo absents : rendu sauté.")
else:
    blender_bin = "/content/blender/blender"
    os.chmod(blender_bin, 0o755)
    env = dict(os.environ, RX="640", RY="270", SPP="4", BACKEND="OPTIX")
    env["DEVICE"] = "GPU" if GPU_OK else "CPU"
    os.chdir("/content/depot")
    t0 = time.time()
    for f in range(1, 97):
        dossier_image = f"{SORTIE}/img_{f:03d}"
        os.makedirs(dossier_image, exist_ok=True)
        env["FRAME"] = str(f)
        r = subprocess.run([blender_bin, "-b", "--factory-startup", "--python", "/content/depot/scenes/chase_mixamo.py", "--", dossier_image],
                           env=env, capture_output=True, text=True)
        png = f"{dossier_image}/chase_a_decor.png"
        if not os.path.exists(png):
            print("ECHEC image", f, r.stderr[-800:])
            break
        os.replace(png, f"{SORTIE}/frame_{f:03d}.png")
        shutil.rmtree(dossier_image)
    print(f"Images : {time.time()-t0:.1f} s ; {env['DEVICE']}")
    subprocess.run(["ffmpeg", "-y", "-loglevel", "error", "-framerate", "24", "-i", f"{SORTIE}/frame_%03d.png",
                    "-c:v", "libx264", "-pix_fmt", "yuv420p", "-crf", "20", f"{SORTIE}/course_sur_place_4s.mp4"], check=True)
    print("MP4 :", f"{SORTIE}/course_sur_place_4s.mp4", round(time.time()-t0, 1), "s")


Images : 647.2 s ; GPU
MP4 : /content/out_mixamo/course_sur_place_4s.mp4 648.7 s
